# D2.8 · Regulatory clock

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.7 · Stop authority](https://spbreed.github.io/cyber-commons/lessons/D2.7.html)**.

| | |
|---|---|
| Tools used | standard library only |

## What this lesson is

**What it covers.** Run the first-hour checklist in a tabletop.

**Why a security engineer needs it.** Notification obligations discovered in week two. The control it builds is: feed Track E2 in hour one.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The disclosure clock starts on the incident, not on your understanding of it. Materiality for a probabilistic actor is genuinely hard, and the hard part does not pause the clock.

> **At CyberTravels.** The disclosure clock started when CyberTravels exported the customer profiles, not when CyberTravels understood what had happened. Passport and payment data make the deadline short. R10.

## 2 · The framework

```
   incident starts -------------------------------> deadline
        |                |                |
     detected        understood        reportable?
                          ^
              materiality for a probabilistic actor is genuinely hard
              and the difficulty does not pause the clock

   trigger criteria are written before, or they are written badly
```

Regulatory clocks start at **awareness** — the point at which you know a
reportable event may have occurred. Not at confirmation, not at containment.

Two consequences that teams discover on day three:

1. **Containing fast does not buy reporting time.** You can contain in an hour
   and still miss a 72-hour deadline, because the clock never paused.
2. **Broken attribution consumes the clock.** If you cannot say who acted
   (D2.1), scoping takes days, and those days are deadline days.

Containment and disclosure are separate workstreams competing for the same
people. If your runbook has one owner for both, one of them is being done badly
under time pressure.

## 3 · The procedure, as a skill

One-hour containment still misses a 72-hour deadline when scoping takes three days, and broken attribution misses it by twenty hours. The skill runs the clock from each candidate awareness point and names the owner of every runbook step.

### The skill — [`skills/response/regulatory-clock-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/regulatory-clock-check/SKILL.md)

```yaml
name: regulatory-clock-check
description: >-
  Run a disclosure deadline from each candidate awareness point and find which
  scoping delays cause it to be missed, then name the owner of each step in the
  runbook. Use when an incident has a reporting obligation and scoping is slow.
allowed-tools: Read, Grep, Glob
```

# The clock started before you knew what happened

Disclosure deadlines run from awareness, and awareness is a defensible judgement
rather than a timestamp. The same incident is met or missed depending on which
point you treat as the start — and the delay that misses it is almost never
containment. It is scoping, and broken attribution is what makes scoping slow.

## When to use this

Any incident with a reporting obligation, and in advance as a tabletop, which is
the only time the answer can still be changed.

## Procedure

**1 — List the candidate awareness points.** First alert, first triage, first
confirmation, first executive notification. Each is arguable, and the earliest
defensible one is the one to plan against.

**2 — Run the clock from each.** Containment, scoping, report drafted, report
submitted. Record met or missed per obligation, per starting point.

**3 — Find the term that misses the deadline.** Containment in an hour and
scoping in three days still misses a 72-hour obligation. Attribution that cannot
say which principal acted turns scoping from hours into days.

**4 — Map obligations to jurisdictions and their clocks.** Different regimes,
different windows, different definitions of a reportable event. One incident can
be inside one and outside another.

**5 — Name an owner per step in the runbook.** Containment, scoping, disclosure
drafting, submission. A runbook with an unowned step is where the hours go.

## Example

**Input** — the fixture committed at the top of [`scripts/regulatory_clock_check.py`](scripts/regulatory_clock_check.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
scenario                            contain   report   met   margin
--------------------------------------------------------------------
fast containment, slow scoping          1.0     80.0 False     -8.0
slow containment, fast reporting       40.0     60.0  True     12.0
both fast                               2.0     20.0  True     52.0
attribution broken (D2.1)               6.0     92.0 False    -20.0

The first row contained in ONE HOUR and still missed the deadline.
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "awareness_points": [{"name": "str", "at": "str", "defensible": true}],
  "obligations": [{"regime": "str", "hours": 0}],
  "runs": [{"from": "str", "containment_h": 0, "scoping_h": 0, "report_h": 0,
            "met": [{"regime": "str", "met": false, "by_hours": 0}]}],
  "dominant_delay": "str",
  "runbook": [{"step": "str", "owner": "str|null"}]
}
```

## Failure modes

- **Starting the clock at confirmation.** A regulator may start it earlier.
- **Optimising containment.** Scoping is the term that misses the deadline.
- **An unowned runbook step.** It is the one that takes a day.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/response/regulatory-clock-check/scripts/regulatory_clock_check.py
SCRIPT = "skills/response/regulatory-clock-check/scripts/regulatory_clock_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan. Both, or the scanning skills clone successfully and then find
    # nothing to look at.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

One-hour containment still misses the 72-hour deadline in the slow-scoping scenario, and broken attribution misses it by 20 hours. The same incident is met or missed depending on which point is treated as awareness. The obligation register shows DORA's 4-hour clock as the binding one, and the runbook check flags a shared owner and a late clock start.

## Your turn

Build your shortest-clock register: every obligation, its deadline, and who notifies. Then check whether your runbook starts the clock at awareness or at confirmation. The gap between those two is often more than a day.

---

**Next → [D2.9 · The fleet kill switch](https://spbreed.github.io/cyber-commons/lessons/D2.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*